In [ ]:
import random
import copy

class Node:
    def __init__(self, floor_state, position: tuple, parent, birth_action, depth):
        self.floor_state = floor_state
        self.position = position
        self.parent = parent
        self.birth_action = birth_action
        self.depth = depth
    
    def get_tuple_floor_state(self):
        return tuple(tuple(row) for row in self.floor_state)

class Stack:
    def __init__(self):
        self.stack = []
    def is_empty(self):
        return len(self.stack) == 0
    def push(self, node: Node):
        self.stack.append(node)
    def pop(self) -> Node:
        return self.stack.pop()
    def contain(self, other: Node):
        for node in self.stack:
            if node.floor_state == other.floor_state and node.position == other.position:
                return True
        return False

GOAL_STATE = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
ROW, COL = 3, 3

def floor_and_vacuumpos_initialize():
    floor = []
    vacuum_pos = (random.randint(0, ROW - 1), random.randint(0, COL - 1))
    for i in range(ROW):
        floor.append([random.randint(0, 1) for _ in range(COL)])
    if floor[vacuum_pos[0]][vacuum_pos[1]] == 1:
        floor[vacuum_pos[0]][vacuum_pos[1]] = 0
    return floor, vacuum_pos

def posible_moves(vacuum_pos):
    moves = []
    if vacuum_pos[0] > 0: moves.append("UP")
    if vacuum_pos[0] < ROW - 1: moves.append("DOWN")
    if vacuum_pos[1] > 0: moves.append("LEFT")
    if vacuum_pos[1] < COL - 1: moves.append("RIGHT")
    return moves

def apply_move(floor, vacuum_pos, move):
    temp_floor = copy.deepcopy(floor)
    if move == "UP": temp_vac_pos = (vacuum_pos[0] - 1, vacuum_pos[1])
    elif move == "DOWN": temp_vac_pos = (vacuum_pos[0] + 1, vacuum_pos[1])
    elif move == "LEFT": temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] - 1)
    else: temp_vac_pos = (vacuum_pos[0], vacuum_pos[1] + 1)
    
    if temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] == 1:
        temp_floor[temp_vac_pos[0]][temp_vac_pos[1]] = 0
    return temp_floor, temp_vac_pos
    
def depth_limit_search(start_floor: list, vacuum_pos: tuple, depth):
    frontier = Stack()
    root = Node(floor_state=start_floor, position=vacuum_pos, birth_action=None, parent=None, depth=0)
    frontier.push(root)
    visited_state = set()
    result = "FAILURE"
    while not frontier.is_empty():
        current_node = frontier.pop()
        
        #kiểm tra và trả về node nếu tìm thấy
        if current_node.floor_state == GOAL_STATE:
            return current_node

        #kiểm tra xem state của node đang xét có trong visited hay không
        if (current_node.get_tuple_floor_state(), current_node.position) not in visited_state:
            visited_state.add((current_node.get_tuple_floor_state(), current_node.position))
            
        if current_node.depth >= depth:
            result = "CUTOFF"
        else:
            moves = posible_moves(current_node.position)
            for move in moves:
                temp_floor, temp_position = apply_move(current_node.floor_state, current_node.position, move)
                temp_node = Node(floor_state=temp_floor, position=temp_position, parent=current_node, birth_action= move, depth= current_node.depth + 1)
                if not frontier.contain(temp_node) and not (temp_node.get_tuple_floor_state() in visited_state):
                    frontier.push(temp_node)
    return result

def iterative_deepening_search():
    floor, vacuum_pos = floor_and_vacuumpos_initialize()
    depth = 0
    
    while True:
        result = depth_limit_search(start_floor=floor, vacuum_pos=vacuum_pos, depth= depth)
        if result != "CUTOFF":
            return result
        depth += 1

def main():
    result = iterative_deepening_search()
    if isinstance(result, Node):
        print(f"\nĐã tìm thấy đáp án (GOAL FOUND)!!!")
        
        # Trích xuất đường đi xuôi dòng từ Root đến Goal
        path = []
        current = result
        while current is not None:
            path.append(current)
            current = current.parent
        path.reverse()
        
        # In chi tiết từng bước di chuyển
        for step, node in enumerate(path):
            if step == 0:
                print(f"Bước xuất phát:")
            else:
                print(f"Bước {step}: Đi [{node.birth_action}]")
            for row in node.floor_state:
                print(row)
            print(f"Vị trí Robot: {node.position}")
            print("-" * 20)
    else:
        print(f"Không tìm thấy đáp án: {result}")

if __name__ == "__main__":
    main()